# TfL New Underground Station Analysis

**LSESU Datathon 2026** — Advising TfL on the optimal location for a new Underground station using data-driven spatial analysis.

**Authors:** [Team Name]  
**Date:** March 2026

## Section 0: Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
from src.config import *
from src.data_download import download_all
from src.data_loader import (
    load_lsoa_boundaries, load_station_locations, load_ptal_grid,
    load_imd_scores, load_census_population, load_census_economic_activity,
)
from src.spatial import build_master_geodataframe
from src.visualisation import (
    plot_distance_to_station, plot_ptal_scores, plot_population_density,
)

print("Imports OK")

## Section 1: Spatial Framework & Data Ingestion (Stage 1)

Build the master LSOA GeoDataFrame that all subsequent stages depend on. This includes:
- Downloading available datasets
- Loading and standardising all data sources
- Computing distance to nearest station and PTAL scores
- Producing choropleth visualisations (V1-V3)

### 1.1 Download Datasets

In [ ]:
# Download datasets with stable URLs (idempotent — skips if already present)
downloaded = download_all()

### 1.2 Load Data

In [ ]:
# Load station locations (always available after download)
stations = load_station_locations()
stations.head()

In [ ]:
# Load LSOA boundaries (requires manual download — see README)
lsoa = load_lsoa_boundaries()
print(f"LSOA columns: {list(lsoa.columns)}")
lsoa.head()

In [ ]:
# Load optional datasets — wrapped in try/except so notebook still runs
# if manual downloads haven't been completed yet

ptal = None
try:
    ptal = load_ptal_grid()
except FileNotFoundError as e:
    print(f"PTAL grid not available: {e}")

imd = None
try:
    imd = load_imd_scores()
except FileNotFoundError as e:
    print(f"IMD scores not available: {e}")

census_pop = None
try:
    census_pop = load_census_population()
except FileNotFoundError as e:
    print(f"Census population not available: {e}")

census_econ = None
try:
    census_econ = load_census_economic_activity()
except FileNotFoundError as e:
    print(f"Census economic activity not available: {e}")

### 1.3 Build Master GeoDataFrame

In [ ]:
# Build master LSOA GeoDataFrame with all joined data
master = build_master_geodataframe(
    lsoa_gdf=lsoa,
    stations_gdf=stations,
    ptal_gdf=ptal,
    imd_df=imd,
    census_pop_df=census_pop,
    census_econ_df=census_econ,
    save=True,
)
print(f"\nMaster GDF shape: {master.shape}")
print(f"Columns: {list(master.columns)}")
master.head()

In [ ]:
# Summary statistics for key columns
master[["dist_to_station_m", "mean_ptal_ai", "area_km2"]].describe()

### 1.4 Stage 1 Visualisations

In [ ]:
# V1: Distance to nearest station
fig1 = plot_distance_to_station(master)
plt.show()

In [ ]:
# V2: PTAL scores (only if PTAL data was loaded)
if "mean_ptal_ai" in master.columns and master["mean_ptal_ai"].notna().any():
    fig2 = plot_ptal_scores(master)
    plt.show()
else:
    print("Skipping V2: PTAL data not available")

In [ ]:
# V3: Population density (only if census data was loaded)
if "pop_density_km2" in master.columns and master["pop_density_km2"].notna().any():
    fig3 = plot_population_density(master)
    plt.show()
else:
    print("Skipping V3: Census population data not available")

---

## Section 2: Demand & Deprivation Scoring (Stage 2)

- Combine population density, IMD deprivation, and economic activity into a composite demand score
- Weight sub-indicators using PCA or domain knowledge
- Identify LSOAs with high unmet demand (high population + high deprivation + low PTAL)

## Section 3: Network Graph Analysis (Stage 3)

- Build Tube/rail network as a weighted graph (NetworkX)
- Compute betweenness centrality, closeness centrality, and degree
- Identify network gaps: areas far from high-centrality nodes
- Evaluate candidate locations for network connectivity improvement

## Section 4: Candidate Site Identification (Stage 4)

- Apply spatial filters: minimum distance from existing stations, within Greater London
- Score candidates using weighted multi-criteria model (demand, connectivity, deprivation)
- Cluster high-scoring LSOAs into candidate zones
- Shortlist top 5-10 candidate sites for detailed evaluation

## Section 5: Impact Modelling (Stage 5)

- Simulate adding each candidate station to the network
- Recompute PTAL, distance-to-station, and catchment population
- Estimate ridership using gravity model with distance decay
- Quantify improvement in accessibility metrics per candidate

## Section 6: Sensitivity Analysis (Stage 6)

- Vary key parameters: catchment radius, decay alpha, scoring weights
- Test robustness of top candidates across parameter ranges
- Monte Carlo simulation for uncertainty quantification
- Identify which candidates are robust vs. parameter-sensitive

## Section 7: Interactive Map (Stage 7)

- Folium interactive map with layers: existing stations, candidate sites, LSOA choropleths
- Pop-up cards for each candidate with key metrics
- Toggle layers for different scoring dimensions
- Export as standalone HTML for presentation

## Section 8: Conclusions & Recommendation (Stage 8)

- Final recommendation with supporting evidence
- Summary table of top candidates with all metrics
- Limitations and caveats
- Suggestions for further analysis (e.g., land use data, cost estimation, engineering feasibility)